# EKF Unicycle Robot Localization

Day 5 notebook framework.

目标：使用 nonlinear unicycle motion model、odometry control 和低频 camera position measurement 完成 EKF 定位。

状态：`x = [px, py, theta]`  
控制：`u = [v, omega]`  
观测：`z = [camera_px, camera_py]`

详细要求见 `notes/day05_ekf_unicycle_model.md`。

## 1. 学习边界和参考代码

今天重点：

- nonlinear motion model
- motion Jacobian
- manual EKF prediction
- camera EKF update
- angle normalization

参考：

- `scripts/03_linear_multisensor_fusion.py`
- `source/filterpy/filterpy/kalman/EKF.py`
- `source/filterpy/filterpy/kalman/tests/test_ekf.py`

不做 EKF-SLAM、IMU bias、landmark range-bearing 或 ROS。

## 2. 导入依赖和设置路径

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from filterpy.kalman import ExtendedKalmanFilter

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FIGURES_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

## 3. 设置实验参数

先使用统一参数跑通，不要同时修改多组参数。

In [ ]:
dt = 0.1
num_steps = 200
initial_state = np.array([0.0, 0.0, 0.0])
true_linear_velocity = 1.0
true_angular_velocity = 0.12

linear_velocity_bias = 0.03
angular_velocity_bias = 0.008
linear_velocity_noise_std = 0.04
angular_velocity_noise_std = 0.01

camera_noise_std = 0.8
camera_interval = 10
dropout_probability = 0.20

position_process_var = 0.01
heading_process_var = 0.005
initial_covariance = 50.0
random_seed = 42

rng = np.random.default_rng(random_seed)

## 4. Angle Normalization

所有 heading 和 heading error 都归一化到 `[-pi, pi)`。

检查：`normalize_angle(3*np.pi)` 应接近 `-np.pi`。

In [ ]:
def normalize_angle(angle):
    # TODO: 将 angle 归一化到 [-pi, pi)
    pass


# 完成函数后取消注释
# test_angles = np.array([0.0, np.pi, 3*np.pi, -3*np.pi])
# normalize_angle(test_angles)

## 5. Nonlinear Unicycle Motion Model

模型：

```text
px' = px + v*cos(theta)*dt
py' = py + v*sin(theta)*dt
theta' = theta + omega*dt
```

输入 state shape `(3,)`，control shape `(2,)`，输出 shape `(3,)`。

In [ ]:
def unicycle_motion_model(state, control, dt):
    # TODO 1: 解包 px, py, theta 和 v, omega
    # TODO 2: 根据非线性模型计算 next_state
    # TODO 3: 归一化 next_state[2]
    # TODO 4: 返回 next_state
    pass


# 完成后测试 theta=0 时是否主要沿 x 前进

## 6. Motion Jacobian

对状态 `[px, py, theta]` 的 Jacobian：

```text
F = [[1, 0, -v*sin(theta)*dt],
     [0, 1,  v*cos(theta)*dt],
     [0, 0,  1]]
```

Jacobian 用于传播协方差，不直接替代 nonlinear state prediction。

In [ ]:
def motion_jacobian(state, control, dt):
    # TODO 1: 读取 theta 和 v
    # TODO 2: 构造 3x3 Jacobian
    # TODO 3: 返回 F
    pass


# 完成后打印 shape，应为 (3, 3)

## 7. Camera Measurement Function and Jacobian

Camera 只测位置：`z = [px, py]`。

`Hx(state)` shape 为 `(2,)`，`HJacobian(state)` shape 为 `(2, 3)`。

In [ ]:
def camera_measurement_function(state):
    # TODO: 返回 state 中的 px, py
    pass


def camera_measurement_jacobian(state):
    # TODO: 返回 shape=(2, 3) 的观测 Jacobian
    pass

## 8. 生成 Ground Truth

创建 constant `v, omega` 控制序列，并使用 nonlinear motion model 逐步生成真实状态。

输出：

- `times`: `(num_steps,)`
- `true_controls`: `(num_steps, 2)`
- `true_states`: `(num_steps, 3)`

In [ ]:
def generate_unicycle_ground_truth(num_steps, dt, initial_state,
                                      linear_velocity, angular_velocity):
    # TODO 1: 创建 times、controls、states
    # TODO 2: 设置 states[0]
    # TODO 3: 从 i=1 开始使用 motion model 积分
    # TODO 4: 返回 times, controls, states
    pass

## 9. 模拟 Odometry Controls

对 `v` 和 `omega` 分别加入 bias 和 Gaussian noise。

输出 `odometry_controls` shape `(num_steps, 2)`。

In [ ]:
def simulate_odometry_controls(true_controls, linear_bias, angular_bias,
                               linear_noise_std, angular_noise_std, rng):
    # TODO 1: 分别生成 v 和 omega noise
    # TODO 2: true control + bias + noise
    # TODO 3: 返回 odometry_controls
    pass


def integrate_odometry_only(initial_state, odometry_controls, dt):
    # TODO: 使用同一个 nonlinear motion model 积分 odometry-only states
    pass

## 10. 模拟 Camera Position Measurements

复用 Day 4 的 interval、noise、dropout 和 availability mask。

无测量位置保持 `NaN`，不要填 `[0, 0]`。

In [ ]:
def simulate_camera_measurements(true_states, noise_std, interval,
                                 dropout_probability, rng):
    num_steps = len(true_states)
    camera_measurements = np.full((num_steps, 2), np.nan)
    camera_available = np.zeros(num_steps, dtype=bool)

    # TODO: 仅在采样且未 dropout 时生成 noisy [px, py]

    return camera_measurements, camera_available

## 11. 创建 Extended Kalman Filter

配置：

- `dim_x=3`
- `dim_z=2`
- `x=[px, py, theta]`
- `P`: `(3, 3)`
- `Q`: `(3, 3)`
- `R`: `(2, 2)`

In [ ]:
def create_ekf(initial_state, initial_covariance, position_process_var,
               heading_process_var, camera_noise_std):
    ekf = ExtendedKalmanFilter(dim_x=3, dim_z=2)

    # TODO 1: 设置 ekf.x 和 ekf.P
    # TODO 2: Q 的前两个对角项使用 position_process_var
    # TODO 3: Q 的 theta 对角项使用 heading_process_var
    # TODO 4: 设置 2x2 camera R

    return ekf

## 12. 运行 EKF Localization

每一步：

1. 保存 `state_before`。
2. 用 `motion_jacobian(state_before, control, dt)` 计算 F。
3. 用 nonlinear motion model 更新 `ekf.x`。
4. 用 `P = F @ P @ F.T + Q` 传播协方差。
5. Camera 可用时调用 `ekf.update(z, HJacobian, Hx)`。
6. 归一化 `theta`。
7. 保存状态。

In [ ]:
def run_ekf_localization(odometry_controls, camera_measurements,
                         camera_available, dt, initial_state,
                         initial_covariance, position_process_var,
                         heading_process_var, camera_noise_std):
    # TODO 1: 创建 EKF 和 states 数组
    # TODO 2: 处理第 0 步 camera update
    # TODO 3: 从 i=1 开始 nonlinear prediction
    # TODO 4: camera 可用时 update，不可用时跳过
    # TODO 5: 每一步归一化 theta 并保存 state
    # TODO 6: 返回 ekf_states
    pass

## 13. 中间检查

完成以上函数后，先检查：

```text
true_states          (200, 3)
true_controls        (200, 2)
odometry_controls    (200, 2)
odometry_states      (200, 3)
camera_measurements  (200, 2)
camera_available     (200,)
ekf_states           (200, 3)
```

并确认全部 heading 在 `[-pi, pi)`。

In [ ]:
# TODO: 完成前面函数后，在此调用并打印所有数组 shape
# 不要在 shape 未确认前直接绘制最终图

## 14. RMSE

需要计算：

- Odometry position RMSE
- EKF position RMSE
- Camera position RMSE（仅有效时刻）
- Odometry heading RMSE
- EKF heading RMSE

Heading error 必须先 normalize。

In [ ]:
def compute_position_rmse(estimated_positions, true_positions):
    # TODO: 复用 Day 4 的二维欧氏距离 RMSE
    pass


def compute_heading_rmse(estimated_heading, true_heading):
    # TODO 1: normalize heading difference
    # TODO 2: 计算 RMSE
    pass


# TODO: 生成 rmse_table 并保存 results/ekf_unicycle_rmse.csv

## 15. 绘图

需要生成：

1. `figures/ekf_unicycle_trajectory.png`
2. `figures/ekf_unicycle_position_error.png`
3. `figures/ekf_unicycle_heading_error.png`

每个函数保存后使用 `plt.close(fig)`。

In [ ]:
def plot_trajectory(true_states, odometry_states, camera_measurements,
                    camera_available, ekf_states, output_path):
    # TODO: 画 ground truth、odometry、camera、EKF
    pass


def plot_position_error(times, true_states, odometry_states,
                        camera_measurements, camera_available,
                        ekf_states, output_path):
    # TODO: 画 odometry、camera、EKF position error
    pass


def plot_heading_error(times, true_states, odometry_states,
                       ekf_states, output_path):
    # TODO: normalize 后画 odometry 和 EKF heading error
    pass

## 16. 验收和结果记录

完成后执行 `Restart Kernel and Run All`，确认：

- Ground truth 是弯曲轨迹。
- Odometry 存在位置和 heading 漂移。
- Camera 稀疏、有噪声和 dropout。
- EKF 轨迹连续并被 camera 修正。
- EKF position RMSE 小于 odometry-only。

然后填写 `notes/day05_ekf_unicycle_model.md` 第 20 节。

## 17. Notebook 跑通后再整理 Python 脚本

目标文件：

`scripts/04_ekf_unicycle_model.py`

整理前检查：

- Notebook 没有核心 TODO。
- 所有 cell 从头运行。
- 三张图和 CSV 已更新。
- 参数、状态顺序、随机种子和时间索引已确认。